In [ ]:
import os
from pathlib import Path 
import allego_file_reader as afr
import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import numpy as np
import math

In [ ]:
import sys
sys.path.append('/Users/ebennett/Library/CloudStorage/Dropbox-EMI/Evelyn Bennett/project_electrodes/electrode_data/electrode-analysis/Code')
import xdat_loader
import figure_fxns


In [ ]:
#reloads if nessesary
import importlib
importlib.reload(xdat_loader)
importlib.reload(figure_fxns)

In [ ]:
# directory to the folder containing the data files
# change to "filname" after done testing
target_dir_augmedia = "/Users/ebennett/Library/CloudStorage/Dropbox-EMI/Evelyn Bennett/project_electrodes/electrode_data/electrode-analysis/Data/20260226_CRC_augmedia" # data directory


In [ ]:
figures_dir = "/Users/ebennett/Library/CloudStorage/Dropbox-EMI/Evelyn Bennett/project_electrodes/electrode_data/electrode-analysis/Figures/20260226_CRC_KCl/AugMedia" # directory to save figures

In [ ]:
# -------- Augmented Media --------

xdat_loader.print_all_xdat_names(target_dir_augmedia)
# creats an array of the indices to feed to the dataframe builder fxn
index_augmedia = xdat_loader.get_xdat_indices(target_dir_augmedia)

In [ ]:
augmedia_df = xdat_loader.build_dataframe(target_dir_augmedia, index_augmedia)


In [ ]:
datasets = augmedia_df.columns.get_level_values('dataset').unique()
for dataset in datasets:
    subset = augmedia_df[dataset].dropna(how='all')
    start_time = subset.index.min()
    end_time = subset.index.max()
    print(f"{dataset}: start = {start_time}, end = {end_time}")

In [ ]:
display_names = [
    "CRC KCl Application Signal",
    "Media Post KCL Signal",
    "CRC Pre KCL Signal",
    "CRC Control Vehcile Signal",
    "Media Pre KCl Signal",
    "CRC Pre KCl Signal",
    "Media Pre 0 (don't use)",
    "CRC Post KCl Signal",
    "Air Control",
    "Media Post KCl Signal"
]

dataset_index = np.arange(0,10)

display_dict = xdat_loader.add_display_names(augmedia_df, dataset_index, display_names)

In [ ]:
#assign color seperately
# assign a color to each file so colors are consistent between graphs
file_colors =xdat_loader.assign_colors(augmedia_df)

In [ ]:
datasets = [
    'media_pre_1__uid0226-15-01-37', 
    'media_post_0__uid0226-16-40-37', 
    'crc_pre_baseline_1__uid0226-15-13-34',
    'crc_vech_ctrl_0__uid0226-15-23-07',
    'crc_kcl_0__uid0226-15-35-33',
]


In [ ]:
channel_id = 3
figure_fxns.raster_plot(
    augmedia_df,
    channel_id,
    datasets=datasets,
    figures_dir=figures_dir
)

In [ ]:
# plot of just kcl application, no ylim
name = 'crc_kcl_0__uid0226-15-35-33'

figure_fxns.plot_channel(
    augmedia_df, 
    channel_id, 
    name, 
    #ylim=((-8000, 9000)),
    figures_dir=figures_dir 
)

In [ ]:
# plot of just kcl application, ylim
name_kcl = 'crc_kcl_0__uid0226-15-35-33'

figure_fxns.plot_channel(
    augmedia_df, 
    channel_id, 
    name_kcl, 
    ylim=((-100, 100)),
    figsize= (12, 6),
    figures_dir=figures_dir 
)

In [ ]:
#--------- Smoothing Function --------
from scipy.signal import savgol_filter, decimate

def smooth_and_decimate(signal, time, fs, 
                        sg_window=31, 
                        sg_poly=3, 
                        decim_factor=100):
    """
    Smooth with Savitzky–Golay filter, then decimate.

    Parameters
    ----------
    signal : 1D numpy array
    fs : sampling frequency (Hz)
    sg_window : Savitzky-Golay window length (must be odd)
    sg_poly : polynomial order (must be < sg_window)
    decim_factor : integer decimation factor (100–1000 typical)

    Returns
    -------
    signal_out : filtered + downsampled signal
    fs_out : new sampling frequency
    """

    # Ensure window is odd
    if sg_window % 2 == 0:
        sg_window += 1

    # Step 1: Savitzky–Golay smoothing
    signal_smooth = savgol_filter(signal, 
                                   window_length=sg_window, 
                                   polyorder=sg_poly)

    # Step 2: Decimate (includes anti-alias filtering internally)
    signal_out = decimate(signal_smooth, 
                          decim_factor, 
                          ftype='iir', 
                          zero_phase=True)
    
    # Decimate the time 
    time_out = time[::decim_factor]

    # Match exact length (sometimes off by 1 due to filtering)
    min_len = min(len(time_out), len(signal_out))
    time_out = time_out[:min_len]
    signal_out = signal_out[:min_len]

    fs_out = fs / decim_factor

    return signal_out, time_out, fs_out


In [ ]:
# ----- Smoothing the KCL application signal -----
fs = 30000  # example original sampling rate (Hz)
signal_crc_kcl = augmedia_df[(name_kcl, channel_id)].dropna().values
time_crc_kcl = augmedia_df[(name_kcl, channel_id)].dropna().index.values.astype(float)  # convert to float seconds
filtered_crc_kcl, filtered_time, new_fs_crc_kcl = smooth_and_decimate(
    signal_crc_kcl,
    time_crc_kcl,
    fs,
    sg_window=71,
    sg_poly=3,
    decim_factor=800
)

print("New sampling rate:", new_fs_crc_kcl)


In [ ]:
colors = augmedia_df.attrs.get('file_colors', {})
plt.figure(figsize=(12,6))
plt.plot(
    filtered_time,
    filtered_crc_kcl,
    color = colors.get(name_kcl, None),
    label = 'CRC KCl Smoothed',
    alpha = 0.7 
)
plt.xlabel('Time (s)')
plt.ylabel('Amplitude (µV)')
plt.title('Electrode 3 -Smoothed CRC KCl Application Signal')
plt.legend()
#plt.xlim(300, 1750)
plt.ylim(-100, 100)

In [ ]:
# ----- apply smoothing to the pbs control signal -----
name_vech_ctrl = 'crc_vech_ctrl_0__uid0226-15-23-07'
signal_vech_ctrl = augmedia_df[(name_vech_ctrl, channel_id)].dropna().values
time_vech_ctrl = augmedia_df[(name_vech_ctrl, channel_id)].dropna().index.values.astype(float)  # convert to float seconds
filtered_vech_ctrl, filtered_time_ctrl, new_fs_vech_ctrl = smooth_and_decimate(
    signal_vech_ctrl,
    time_vech_ctrl,
    fs,
    sg_window=71,
    sg_poly=3,
    decim_factor=800
)

print("New sampling rate:", new_fs_crc_kcl)


In [ ]:
figure_fxns.plot_channel(
    augmedia_df, 
    channel_id, 
    name_vech_ctrl, 
    ylim=((-100, 100)),
    figsize= (12, 6),
    figures_dir=figures_dir 
)

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(
    filtered_time_ctrl,
    filtered_vech_ctrl,
    color = colors.get(name_vech_ctrl, None),
    label = 'CRC Vech Ctrl Smoothed',
    alpha = 0.7 
)
plt.xlabel('Time (s)')
plt.ylabel('Amplitude (µV)')
plt.title('Electrode 3 - Smoothed CRC Vech Ctrl Application Signal')
plt.legend()
#plt.xlim(300, 1750)
plt.ylim(-100, 100)

Noise Analysis

In [ ]:
start = 350
end = 600
idx_start = np.argmin(np.abs(time_crc_kcl - start))
idx_end = np.argmin(np.abs(time_crc_kcl - end))

print(f"Start index: {idx_start}, End index: {idx_end}")

In [ ]:
kcl_noise_signal = signal_crc_kcl[idx_start:idx_end]
noise_time = time_crc_kcl[idx_start:idx_end]

In [ ]:
#num_bins = math.ceil(np.sqrt(len(signal_crc_kcl)))
# this is 4437 that is way to high
num_bins = math.ceil(np.log2(len(signal_crc_kcl)) + 1)
print(f"Number of bins for KCl noise signal: {num_bins}")

In [ ]:
hist_kcl, bins_kcl = np.histogram(kcl_noise_signal, num_bins, density=True)
bin_centers_kcl = 0.5 * (bins_kcl[1:] + bins_kcl[:-1])

In [ ]:
# crc pre gaussian fit before spike segment
from scipy.optimize import curve_fit

def gaussian(x, A, mu, sigma):
    return A * np.exp(-0.5 * ((x - mu) / sigma)**2)

A_guess = np.max(hist_kcl)
mu_guess = np.mean(kcl_noise_signal)
sigma_guess = np.std(kcl_noise_signal)

popt_kcl, pcov = curve_fit(gaussian, bin_centers_kcl, hist_kcl, p0=[A_guess, mu_guess, sigma_guess])
A_fit, mu_fit, sigma_fit_kcl = popt_kcl
print(f"Fitted parameters: A = {A_fit}, mu = {mu_fit}, sigma = {sigma_fit_kcl}")

In [ ]:
#compute FWHM
fwhm_kcl = 2 * np.sqrt(2 * np.log(2)) * sigma_fit_kcl
print(f"FWHM: {fwhm_kcl}")

In [ ]:
#plot histogram and fit for pre spike segment
plt.figure(figsize=(16, 6))

#origonal signal plot
plt.subplot(121)
plt.plot(noise_time, 
         kcl_noise_signal, 
         alpha=0.7, 
         label='CRC Post-KCl Dose 1', 
         color=colors.get(name_kcl, None)
        )
plt.ylim(-50, 50)
plt.title('CRC Post-KCl Noise in Growth Media')
plt.xlabel('Time (s)')
plt.ylabel('Signal Amplitude (µV)')

#guassian plot
plt.subplot(122)
plt.hist(kcl_noise_signal, bins=num_bins, density=True, alpha=0.6, color='g', label='Signal Histogram')
x_fit = np.linspace(np.min(kcl_noise_signal), np.max(kcl_noise_signal), 1000)
y_fit = gaussian(x_fit, *popt_kcl)
plt.plot(x_fit, y_fit, 'r--', label='Gaussian Fit', color="black")
plt.plot([], [], ' ', label=f'FWHM = {fwhm_kcl:.3f}')
plt.xlim(-50, 50)
plt.title('Gaussian Fit of CRC Noise in Growth Media')
plt.xlabel('Signal Amplitude (µV)')
plt.ylabel('Probability Density')
plt.legend(loc='upper left')

